# M0. Efficiency considerations

Dendra supports TorchDynamo / Inductor compilation, but JIT is not enabled globally by default. Enable it when a workload is large enough to amortize compilation overhead.

## JIT compilation overhead

JIT compilation introduces overhead during the first few simulation steps. For long simulations, this overhead is usually negligible, but for short simulations, small batches, or highly interactive development, eager mode can be faster.

```python
import dendra as dn

with dn.ctx(JIT=1):
    model = YourModel()
    model.run(tstop=100.0, dt=0.025)
```

You can also enable or disable broad JIT globally:

```python
dn.set_jit_enabled(True)
dn.set_jit_enabled(False)
```

Networks have more granular switches:

```python
with dn.ctx(JIT_NETWORK_SOLVES=1):
    # compile population / integrator solves inside networks
    ...

with dn.ctx(JIT_NETWORK_OPS=1):
    # compile network-side event / synapse plumbing where supported
    ...
```

Use `JIT_NETWORK_SOLVES` first for large biophysical populations. Use `JIT_NETWORK_OPS` when network-side event delivery dominates and your workload is stable enough to benefit from compilation.

## Network object count

For event-driven networks, Python and tensor-launch overhead often scale with the number of NetCon groups and mechanism objects. Prefer:

- banked point-process mechanisms with `insert(..., copies=...)`;
- explicit `SynapseSlots` targets;
- grouped connections with one `NetCon` per homogeneous source/target/synapse class block;
- shared `spikedetect` mechanisms with `threshold=None` for uniform thresholds;
- static-zero connection omission when the weights are not trainable.

The goal is to preserve the scientific unit of independence, such as one local synaptic slot per contact, without preserving one Python object per contact.

## PyTorch version recommendations

Dendra targets Python 3.11+ and PyTorch 2.12+ for the current documentation examples.

For small-batch CPU simulations, explicitly limiting PyTorch threading can improve performance:

```python
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

import dendra as dn
```

For GPU/MPS workloads, construct directly on the intended device and dtype with `dn.ctx(DEVICE=..., DTYPE=...)` to avoid large post-construction copies.